# 05. Regimes and Structural Breaks

This is the core notebook of the project. The earlier notebooks established whether volatility, dependence and transmission changed after COVID-19. Here we ask the strongest version of the research question: **did the market move into distinct regimes, and do structural breaks cluster around the COVID period?**

We focus on two objects that summarize the structure of risk particularly well:
- the rolling variance of the S&P 500, as a direct proxy for aggregate market risk;
- the rolling correlation between the S&P 500 and the US 10Y yield change, as a direct proxy for the equity/rates risk structure.

## 1. Setup

The notebook combines two complementary approaches from the course. A Markov-switching model asks whether the equity/rate dependence alternates between latent regimes. The Bai-Perron style multiple-break search asks whether the same series display discrete structural breaks at particular dates.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project2_regime_utils import (
    load_regime_data,
    fit_two_state_markov,
    markov_summary_table,
    smoothed_probability_table,
    plot_smoothed_probabilities,
    run_break_detection,
    plot_breaks,
    pre_post_break_count,
    save_regime_outputs,
)
from project2_data_utils import ensure_output_dirs, save_figure

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.4f}".format
ensure_output_dirs()

## 2. Build the two regime-state variables

We first construct the two series that will carry the regime analysis:
- a rolling annualized S&P 500 variance;
- a rolling 52-week SPX / UST rolling correlation.

These are natural state variables because they summarize both the level of risk and the way risk is transmitted between equities and sovereign yields.

In [ ]:
aligned_returns, rolling_variance, spx_ust_corr = load_regime_data()

print(f"Rolling variance sample: {rolling_variance.shape}")
print(f"Rolling correlation sample: {spx_ust_corr.shape}")

rolling_variance.head(), spx_ust_corr.head()

## 3. Two-state Markov switching on the SPX / UST correlation

The Markov-switching model is estimated on the rolling SPX / UST correlation. Economically, the idea is simple: if the structure of risk changed, then the equity/rate relation may switch between a regime where stocks and yields move together and a regime where they move differently, for example because one of them plays a hedging role during stress.

In [ ]:
markov_result = fit_two_state_markov(spx_ust_corr)
markov_summary = markov_summary_table(markov_result)
markov_summary

## 4. Smoothed regime probabilities

The smoothed probabilities tell us when the model believes the market is more likely to be in one regime rather than the other. This is often more informative than the raw parameter table because it shows whether the COVID period coincides with a persistent shift in the inferred state of the market.

In [ ]:
smoothed_probs = smoothed_probability_table(markov_result)
smoothed_probs.head()

In [ ]:
probability_figure = plot_smoothed_probabilities(smoothed_probs)
save_figure(probability_figure, "05_markov_smoothed_probabilities.png")
plt.show()

## 5. Bai-Perron style breaks on S&P 500 rolling variance

A latent-regime model is useful, but it does not tell us whether the data also contain sharp discrete breaks. We therefore run a Bai-Perron style multiple-break detection on the rolling variance of the S&P 500. If several break dates cluster around COVID or later, that is direct evidence that the risk process changed structurally rather than only temporarily.

In [ ]:
variance_series = rolling_variance.set_index("date")["sp500_rolling_variance"]
variance_breaks = run_break_detection(variance_series)
variance_breaks

In [ ]:
variance_break_figure = plot_breaks(
    variance_series,
    variance_breaks,
    title="Figure 2. Bai-Perron style breaks in S&P 500 rolling variance",
    ylabel="Annualized rolling variance",
)
save_figure(variance_break_figure, "05_sp500_variance_breaks.png")
plt.show()

## 6. Bai-Perron style breaks on the SPX / UST rolling correlation

We repeat the same exercise on the rolling SPX / UST correlation. This is especially important because the project is not only about more or less volatility. It is also about whether the relationship between risky assets and sovereign rates was reorganized after COVID.

In [ ]:
correlation_series = spx_ust_corr.set_index("date")["spx_ust_rolling_corr"]
correlation_breaks = run_break_detection(correlation_series)
correlation_breaks

In [ ]:
correlation_break_figure = plot_breaks(
    correlation_series,
    correlation_breaks,
    title="Figure 3. Bai-Perron style breaks in SPX / UST rolling correlation",
    ylabel="Rolling correlation",
)
save_figure(correlation_break_figure, "05_spx_ust_corr_breaks.png")
plt.show()

## 7. Count how breaks are distributed around COVID

A simple but useful summary is to count how many estimated breaks fall before and after the COVID date. This is not a test by itself, but it helps translate the break analysis into a direct answer to the research question.

In [ ]:
variance_break_counts = pre_post_break_count(variance_breaks)
correlation_break_counts = pre_post_break_count(correlation_breaks)

print("Break counts for S&P 500 rolling variance")
variance_break_counts

In [ ]:
print("Break counts for SPX / UST rolling correlation")
correlation_break_counts

## 8. Save the regime and break outputs

We save the Markov summary, the smoothed probabilities and the two break tables because these objects are the core evidence for the final synthesis notebook.

In [ ]:
save_regime_outputs(markov_summary, smoothed_probs, variance_breaks, correlation_breaks)
print("Saved regime and break outputs under outputs/project2/")

## 9. Main takeaways from notebook 05

This notebook provides the strongest direct evidence on whether the structure of risk changed after COVID-19. If the Markov model shows persistent switches in the probability of the lower- or higher-correlation regime, and if the break detection finds multiple breaks in the S&P 500 variance or SPX / UST dependence around the COVID period, then the case for a structural post-COVID reorganization of market risk becomes much stronger.